# 06 · Escala: `nlp.pipe` y `DocBin`

## Propósito

Hasta acá corriste todo sobre muestras de unos cientos de párrafos. El corpus
tiene 5.428 y son 2,4 millones de caracteres.

Este es el cuaderno que convierte a spaCy de demostración en herramienta de
trabajo: cómo procesar un corpus entero, qué se puede apagar, y —lo más
importante— cómo no volver a procesarlo nunca más.

## Qué consume

- `datos/corpus.jsonl`
- `pipeline_propio/` y `componentes.py`

## Qué produce

- `datos/corpus.spacy`

## ◈ Microglosario

| Término | Qué es |
|---|---|
| **`nlp.pipe()`** | Procesa muchos textos de una. No es lo mismo que llamar a `nlp()` en un ciclo |
| **lote** (*batch*) | El grupo de textos que `pipe` manda junto en cada vuelta. Lo controlás con `batch_size` |
| **`disable`** | La lista de componentes que apagás para esta corrida. Cada uno que apagues se lleva lo que dependía de él |
| **`DocBin`** | Un contenedor que guarda `Doc` ya procesados en un archivo binario, con todas sus anotaciones adentro |
| **serializar** | Convertir algo que está en memoria en algo que se puede escribir en disco y volver a leer |

## Celda de reanudación

Fijate en la línea `import componentes`. Sin ella, `spacy.load("pipeline_propio")`
falla: spaCy guardó **qué** componentes tiene el pipeline, pero no su código.

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# time: para medir cuánto tarda cada cosa.
import time

# Nuestros componentes propios. Al importarlos, quedan registrados en spaCy.
import componentes

# pipeline_propio: la herramienta que configuraste y guardaste en el cuaderno 05
nlp = spacy.load("pipeline_propio")
# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

# pipe_names: la lista de componentes que tiene el pipeline cargado
print("pipeline:", nlp.pipe_names)
cantidad_parrafos = len(corpus)
# :, separa los miles con una coma
print("párrafos:", f"{cantidad_parrafos:,}")

if "corrector_voseo" not in nlp.pipe_names:
    print("ATENCIÓN: el pipeline_propio no tiene el corrector de voseo. Volvé a correr el 05")
    print("hasta el final: es el cuaderno que guarda la herramienta en disco.")

## 1. `nlp.pipe()`: procesar muchos documentos

Lo venís usando desde el `01`. En vez de un `for` que llama a `nlp()` en cada
texto, `nlp.pipe()` agrupa el trabajo en lotes, y el tamaño de cada lote lo
controlás con `batch_size`.

La ganancia de velocidad existe, pero es modesta: **`nlp.pipe` no es el truco de
velocidad que suele venderse.** Lo que sí conviene saber es qué devuelve, porque
ahí está la trampa.

### El detalle que rompe: `nlp.pipe` devuelve un generador

Esto es importante y no es obvio.

In [ ]:
primeros = corpus.head(600)
muestra = list(primeros["texto"])

resultado = nlp.pipe(muestra)

# type(): qué clase de objeto es
print(type(resultado))

Un **generador** produce los documentos de a uno, a medida que los recorrés.
No los guarda todos.

La tentación es escribir `documentos = list(nlp.pipe(textos))` para poder
recorrerlos varias veces. Con párrafos, como los de nuestro corpus, entra en
memoria sin problema, y más adelante lo vamos a hacer. Con documentos grandes
—los doce archivos originales, uno por año— **agota la memoria y el kernel se
interrumpe sin dar un mensaje claro.**

**Regla práctica:** convertí el generador en lista solo si trabajás con unidades
chicas. Si necesitás recorrer documentos grandes más de una vez, la solución
correcta es guardarlos con `DocBin`, en la sección 3.

## 2. `disable`: apagar lo que no usás

El pipeline tiene nueve componentes. Si tu pregunta no los necesita a todos,
estás pagando por nada.

In [ ]:
# Cada configuración dice qué componentes apagamos.
configuraciones = {
    "todo": [],
    "sin parser": ["parser"],
    "sin parser ni ner": ["parser", "ner"],
    "solo tokenizar": ["morphologizer", "parser", "attribute_ruler",
                       "lemmatizer", "entity_ruler", "ner",
                       "corrector_voseo", "detector_personajes"],
}

# Un for sobre un diccionario recorre sus claves: acá, los nombres de las configuraciones.
for nombre in configuraciones:
    componentes_apagados = configuraciones[nombre]
    inicio = time.time()
    # disable=: apaga estos componentes mientras procesa
    for documento in nlp.pipe(muestra, batch_size=50, disable=componentes_apagados):
        pass
    demora = time.time() - inicio
    demora_redondeada = round(demora, 1)  # round(..., 1): con un decimal
    print(" ", nombre, "->", demora_redondeada, "s")

El `parser` es el componente caro, y tiene sentido: es el que construye el
árbol sintáctico.

> ### Perilla
>
> **La lista de `disable` no es una optimización técnica: es la declaración de
> qué le estás preguntando al texto.**
>
> - ¿Contás sustantivos por frecuencia? No necesitás el `parser`.
> - ¿Buscás nombres propios de una lista? Te alcanza con el `entity_ruler`.
> - ¿Analizás quién es sujeto de qué verbo? Ahí sí necesitás el `parser`, y no
>   hay atajo.
>
> Pero cuidado con una trampa: **desactivar el `parser` también te deja sin
> `doc.sents` y sin `noun_chunks`**, porque los dos dependen del árbol.
> Comprobalo.

In [ ]:
# Sin el parser, doc.sents no existe. Esta celda falla a propósito.
documento_sin_parser = nlp("Una oración. Y otra oración distinta.", disable=["parser"])

list(documento_sin_parser.sents)

El error lo dice sin vueltas: sin el `parser`, el `Doc` no tiene límites de
oración, y `doc.sents` no tiene de dónde sacarlos.

Por eso la lista de `disable` no es una decisión de velocidad. Cada componente que
apagás se lleva con él todo lo que dependía de él.

## 3. `DocBin`: procesar una vez, guardar, recargar

Acá está la mejora que de verdad cambia cómo se trabaja.

`DocBin` guarda documentos **ya procesados** en un archivo binario. Todas las
anotaciones viajan adentro: tokens, lemas, POS, morfología, dependencias,
entidades.

Vamos a procesar el corpus completo. Tarda alrededor de un minuto.

In [ ]:
# DocBin: el contenedor de documentos procesados.
from spacy.tokens import DocBin

# os: para mirar el tamaño del archivo.
import os

textos = list(corpus["texto"])

# store_user_data=True guarda también nuestras extensiones (doc._.personajes).
banco = DocBin(store_user_data=True)

inicio = time.time()
# batch_size=100: cada lote lleva 100 textos
for documento in nlp.pipe(textos, batch_size=100):
    banco.add(documento)  # add(): suma el documento procesado al banco
# to_disk(): escribe el banco entero en un archivo
banco.to_disk("datos/corpus.spacy")
procesar_y_guardar = time.time() - inicio

# getsize(): el tamaño del archivo, en bytes
bytes_del_archivo = os.path.getsize("datos/corpus.spacy")
megabytes_del_archivo = bytes_del_archivo / 1_000_000
cantidad_textos = len(textos)
print(f"procesar {cantidad_textos:,} párrafos y guardar: {procesar_y_guardar:.1f} s")
print(f"archivo: {megabytes_del_archivo:.1f} MB")

In [ ]:
inicio = time.time()

# Creamos un banco vacío y lo llenamos con lo que guardamos en disco.
banco_leido = DocBin()
banco_leido.from_disk("datos/corpus.spacy")

# get_docs() pide el vocab del pipeline: cada Doc de spaCy necesita la tabla de
# palabras del modelo para volver a armarse.
generador = banco_leido.get_docs(nlp.vocab)
documentos = list(generador)

recargar = time.time() - inicio

entidades = 0
for documento in documentos:
    # .ents: las entidades que encontró el NER en ese documento
    entidades = entidades + len(documento.ents)

cantidad_documentos = len(documentos)
veces_mas_rapido = procesar_y_guardar / recargar

print(f"recargar desde disco: {recargar:.1f} s")
print(f"documentos: {cantidad_documentos:,} | entidades intactas: {entidades:,}")
print()
print(f"veces más rápido que reprocesar: {veces_mas_rapido:.0f}x")

Y las anotaciones están todas, incluidas las nuestras.

In [ ]:
# Buscamos el primer párrafo donde nuestro detector encontró personajes.
documento = None
for candidato in documentos:
    # Si no encontró ninguno, queda vacío, y lo vacío cuenta como falso para el if.
    if candidato._.personajes:
        documento = candidato
        break  # break: cortamos el for, ya encontramos uno

# [:80]: los primeros 80 caracteres
inicio_del_texto = documento.text[:80]
print("texto:", inicio_del_texto, "...")

print()
print("lemas de los primeros seis tokens:")
# documento[:6]: los primeros seis tokens
for token in documento[:6]:
    print("  ", token.text, "->", token.lemma_)

print()
print("primeras entidades:")
# ents[:4]: las primeras cuatro entidades
for entidad in documento.ents[:4]:
    print("  ", entidad.text, "->", entidad.label_)

print()
print("personajes:", documento._.personajes)

Fijate que `documento._.personajes` volvió como **tupla** y no como lista:
así serializa spaCy los datos de usuario. Es un detalle menor pero te va a
dar problemas si tu código hace `.append()` sobre lo que recuperaste del `DocBin`.

### `to_disk()` y `DocBin` no son lo mismo

Los dos escriben en disco y se confunden todo el tiempo. Guardátelo así:

| | qué guarda | cuándo lo usás |
|---|---|---|
| `nlp.to_disk("pipeline_propio")` | **La herramienta.** El modelo, las reglas, los componentes. | Cuando configuraste un pipeline y querés reusarlo o compartirlo. |
| `DocBin.to_disk("corpus.spacy")` | **Los resultados.** Los documentos ya procesados. | Cuando procesaste un corpus y no querés volver a hacerlo. |

Con los dos archivos, cualquiera puede reproducir tu análisis sin repetir el
trabajo: uno trae la herramienta, el otro trae la salida.

## 4. El corpus completo, ahora sí

Con todo procesado, una pregunta que antes era cara ahora es instantánea.
Contemos entidades sobre los 5.428 párrafos.

In [ ]:
# Counter: para contar.
from collections import Counter

conteo_personajes = Counter()
conteo_lugares = Counter()

for documento in documentos:
    for entidad in documento.ents:
        if entidad.label_ == "PERSONAJE":
            conteo_personajes[entidad.text] += 1
        # elif: si no era PERSONAJE, probamos con LUGAR
        elif entidad.label_ == "LUGAR":
            conteo_lugares[entidad.text] += 1

print("--- personajes en todo el corpus ---")
# most_common(10): los 10 más frecuentes, de mayor a menor
for nombre, veces in conteo_personajes.most_common(10):
    print(" ", nombre, "->", veces)

print()
print("--- lugares ---")
for nombre, veces in conteo_lugares.most_common(8):
    print(" ", nombre, "->", veces)

In [ ]:
# Y ahora por año, que es la única variable temporal que tiene el corpus.
anios = list(corpus["anio"])

filas = []
# posicion: en qué párrafo vamos. Los documentos están en el mismo orden que el
# corpus, así que en esa misma posición de anios está su año.
posicion = 0
for documento in documentos:
    anio = anios[posicion]
    for entidad in documento.ents:

        # Solo nos interesan las entidades marcadas como PERSONAJE.
        if entidad.label_ == "PERSONAJE":
            fila = {"anio": anio, "personaje": entidad.text}
            filas.append(fila)
    posicion = posicion + 1

apariciones = pd.DataFrame(filas)

# crosstab(): tabla cruzada. Cuenta cuántas veces aparece cada personaje (index, las
# filas) en cada año (columns, las columnas).
pd.crosstab(index=apariciones["personaje"], columns=apariciones["anio"])

## 5. Ejercicio

1. Con el corpus ya procesado, contá los verbos más frecuentes **por año**
   usando lemas. ¿Se nota el efecto del `corrector_voseo`?
2. ¿Qué proporción de párrafos no contiene ninguna entidad? ¿Son párrafos cortos?
3. Volvé a generar el `DocBin` con `disable=["parser"]` y compará el tamaño del
   archivo. ¿Qué se perdió?

In [ ]:
# TU CÓDIGO ACÁ
#
# Punto 2: la proporción de párrafos sin entidades.
#
#   Paso 1: arrancá dos contadores en cero: sin_entidades y total.
#
#   Paso 2: recorré documentos. En cada vuelta:
#           - sumale 1 a total;
#           - guardá len(documento.ents) en una variable;
#           - si esa variable da 0, sumale 1 a sin_entidades.
#
#   Paso 3: proporcion = sin_entidades / total
#
#   Paso 4: para ver si son cortos, juntá en una lista el len(documento.text) de
#           los párrafos sin entidades, y compará su promedio con
#           corpus["caracteres"].mean()
#
# Punto 3: repetí la celda del DocBin con disable=["parser"], pero guardalo como
#              "datos/corpus_sin_parser.spacy"
#          Si lo guardás como "datos/corpus.spacy", pisás el archivo que carga
#          el cuaderno 07.

## Lo que hiciste hoy

- Aprendiste que `nlp.pipe` devuelve un generador y por qué convertirlo en lista
  puede agotar la memoria.
- Apagaste componentes y viste el costo de cada uno, y también qué se pierde.
- Procesaste el corpus completo una vez y lo guardaste con `DocBin`.
- Distinguiste `to_disk()` (la herramienta) de `DocBin` (los resultados).

## Lo que viene

En el `07`, la última capa: los vectores de palabras. Es también el puente hacia
lo que sigue en la materia.